<a href="https://colab.research.google.com/github/asmaslenikova/maslenikova-compling/blob/main/%D0%9C%D0%B0%D1%81%D0%BB%D0%B5%D0%BD%D0%B8%D0%BA%D0%BE%D0%B2%D0%B0_%22w2v_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [3]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.6 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [4]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

**Базовые операции с векторами**

In [5]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [6]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [7]:
w2v_model = api.load('glove-wiki-gigaword-100')

2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [8]:
def top10(word):
    try:
        return w2v_model.most_similar(word, topn=10)
    except KeyError:
        return None

3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [9]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [24]:
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

In [12]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [25]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [29]:
# Найдите слова, похожие на "духовка"
### ваш код здесь ###

similar_duhovka = model.wv.most_similar('духовка', topn=5)
print(similar_duhovka)


# Найдите слова, похожие на "овощи"
### ваш код здесь ###
similar_ovoshi = model.wv.most_similar('овощи', topn=5)
print(similar_ovoshi)

[('ингредиенты', 0.3198968470096588), ('десерт', 0.30644166469573975), ('холодильник', 0.27045494318008423), ('питание', 0.22426293790340424), ('пирог', 0.21422232687473297)]
[('мариновать', 0.2715907096862793), ('хлеб', 0.26912084221839905), ('гриль', 0.25464725494384766), ('фольга', 0.24094568192958832), ('сахар', 0.21084155142307281)]


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [21]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [30]:
ft_varit = ft_model.wv.most_similar('варить', topn=5)
print(ft_varit)
ft_duhovka = ft_model.wv.most_similar('духовка', topn=5)
print(ft_duhovka)
ft_ovoshi = ft_model.wv.most_similar('овощи', topn=5)
print(ft_ovoshi)

[('жарить', 0.5353310108184814), ('парить', 0.4804992079734802), ('месить', 0.3540874421596527), ('тушить', 0.34048697352409363), ('специи', 0.2621820569038391)]
[('взбивать', 0.45650428533554077), ('лимон', 0.3561227023601532), ('салат', 0.30499011278152466), ('курица', 0.3041205108165741), ('тост', 0.2943783700466156)]
[('жарить', 0.29603469371795654), ('фольга', 0.25739818811416626), ('морковь', 0.2296556979417801), ('соус', 0.217234805226326), ('торт', 0.20936326682567596)]


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [31]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models("варть")  # опечатка
compare_models("овожи")  # опечатка
compare_models("духофка")  # опечатка


Сравнение для слова: 'варть'
  Word2Vec: слово не найдено
  FastText: ['морковь', 'дрожжи']

Сравнение для слова: 'овожи'
  Word2Vec: слово не найдено
  FastText: ['барбекю', 'говядина']

Сравнение для слова: 'духофка'
  Word2Vec: слово не найдено
  FastText: ['бекон', 'травы']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [32]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [33]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [34]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [35]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [36]:
sim_2_4 = doc_model.dv.similarity("doc_2", "doc_4")
print("Сходство:", sim_2_4)

Сходство: -0.0362442


9. Найдите самый похожий документ на doc_1

In [37]:
print(doc_model.dv.most_similar("doc_1", topn=1))

[('doc_0', 0.2735169529914856)]


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [38]:
results = {}

for size in [10, 50, 100]:
    m = Word2Vec(
        cooking_sentences,
        vector_size=size,
        window=3,
        min_count=1,
        workers=2,
        sg=1
    )
    results[size] = {
        "варить": m.wv.most_similar("варить", topn=3),
        "овощи": m.wv.most_similar("овощи", topn=3),
        "духовка": m.wv.most_similar("духовка", topn=3),
    }

print(results)

{10: {'варить': [('рыба', 0.6653584837913513), ('сковорода', 0.6197580099105835), ('хлеб', 0.6120493412017822)], 'овощи': [('жарить', 0.7185095548629761), ('фольга', 0.7146145105361938), ('горшок', 0.6696175336837769)], 'духовка': [('взбивать', 0.5916136503219604), ('тушить', 0.5880028009414673), ('говядина', 0.5443191528320312)]}, 50: {'варить': [('вино', 0.23975302278995514), ('ингредиенты', 0.21724146604537964), ('хлеб', 0.1937645971775055)], 'овощи': [('мариновать', 0.2715907096862793), ('хлеб', 0.26912084221839905), ('гриль', 0.25464725494384766)], 'духовка': [('ингредиенты', 0.3198968470096588), ('десерт', 0.30644166469573975), ('холодильник', 0.27045494318008423)]}, 100: {'варить': [('чашка', 0.3190118670463562), ('вино', 0.20387358963489532), ('сковорода', 0.1985575407743454)], 'овощи': [('взбивать', 0.21899110078811646), ('питание', 0.21628890931606293), ('бекон', 0.1956300586462021)], 'духовка': [('жарить', 0.19041824340820312), ('парить', 0.16695661842823029), ('завтрак', 0.